# M01 — 對話模型與訊息

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

你會練到：
- chat model 的統一介面：`invoke` / `stream` / `batch`
- 四種訊息物件：`SystemMessage` / `HumanMessage` / `AIMessage` / `ToolMessage`
- content blocks 與多模態（文字 + 圖片）的訊息結構
- 用 `temperature` 調行為、用 `model.profile` 查模型能力

注意：本 notebook 的程式碼仰賴你已在 repo 根目錄設好 `.env`（見 M00）。

## 1. 環境準備

載入課程共用 helper，並透過 `get_model()` 拿到一顆「供應商無關」的模型。
換 OpenAI / Anthropic / Ollama 只需改環境變數，這格程式碼不用動。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. invoke：送一次、等完整回答

`invoke` 的輸入可以只是一個字串。回傳是一個 `AIMessage`，取文字用 `.content`。
這是最常用、最簡單的形式。

In [ ]:
response = model.invoke("用一句話介紹你自己")
print(type(response).__name__)   # Expected output: AIMessage
print(response.content)          # Expected output: 一句模型的自我介紹

## 3. 用訊息物件組一段對話

真實對話有角色之分。最常見的組合：`SystemMessage` 設定規則 + `HumanMessage` 提問。
訊息一律從 `langchain.messages` 匯入。

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("你是嚴謹的物理老師，只用繁體中文、兩句話以內回答"),
    HumanMessage("什麼是慣性？"),
]
response = model.invoke(messages)
print(response.content)   # Expected output: 用兩句話以內解釋慣性

## 4. dict 簡寫形式

每個訊息物件都等價於一個 `{"role": ..., "content": ...}` 的 dict。
下面這段和上一格效果相同；兩種寫法可混用。

In [ ]:
messages_as_dict = [
    {"role": "system", "content": "你是嚴謹的物理老師，只用繁體中文、兩句話以內回答"},
    {"role": "user", "content": "什麼是慣性？"},
]
response = model.invoke(messages_as_dict)
print(response.content)   # Expected output: 與上一格類似的答案

## 5. AIMessage：手動塞入對話歷史

你也可以親手放入 `AIMessage`，模擬「之前模型說過的話」，讓這次提問接得上下文。
`ToolMessage` 則用來放工具執行結果，會在 M04 工具呼叫時大量登場，這裡先認識它的存在。

In [ ]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage("你是物理老師，回答簡短"),
    HumanMessage("牛頓第一定律是什麼？"),
    AIMessage("物體在不受外力時會保持靜止或等速直線運動。"),
    HumanMessage("那第二定律呢？"),   # follow-up relies on the history above
]
response = model.invoke(conversation)
print(response.content)   # Expected output: 解釋 F = ma 的簡短說明

## 6. stream：逐塊吐回（打字機效果）

`stream` 把同一次回答切成很多 `AIMessageChunk`。
每塊只是片段，把 `.content` 接起來才是完整答案。

In [ ]:
print("串流輸出：", end="")
for chunk in model.stream("用三句話講為什麼天空是藍色的"):
    # Each chunk is a partial piece, not the whole answer.
    print(chunk.content, end="", flush=True)
print()   # newline at the end
# Expected output: 文字一塊一塊地印出，最後組成完整段落

## 7. batch：一次送多筆、平行處理

多筆「彼此獨立」的輸入，用 `batch` 一次送出，比一筆筆 `invoke` 更省時。
回傳是一個 `AIMessage` 的 list，順序對應輸入。

In [ ]:
questions = [
    "用一句話說明什麼是質量",
    "用一句話說明什麼是重量",
    "用一句話說明什麼是密度",
]
answers = model.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q}\nA: {a.content}\n")
# Expected output: 三組問答依序印出

## 8. content blocks 與多模態（概念示範）

關鍵觀念：一則訊息的 `content` 不一定是字串，也可以是「內容區塊」的 list。
每個區塊用 `type` 標明是文字還是圖片——這就是多模態的基礎。

下面只是**示範訊息的資料結構**，不直接送出（能否看圖取決於模型本身）。
純文字的 `content` 其實只是「只有一個 text 區塊」的簡寫。

In [ ]:
# A multimodal HumanMessage: text + image in the SAME message.
multimodal_message = HumanMessage(content=[
    {"type": "text", "text": "這張圖裡有什麼動物？"},
    {
        "type": "image",
        "source_type": "url",
        "url": "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg",
    },
])

# Inspect the structure instead of sending it.
print("content 是 list 嗎？", isinstance(multimodal_message.content, list))  # Expected: True
for block in multimodal_message.content:
    print("block type:", block["type"])
# Expected output: block type: text / block type: image

## 9. 調參數：temperature

參數在 `get_model()` 時傳入即可（會原封不動轉給 `init_chat_model`）。
`temperature=0` 最穩定、可重現；越高越有創意但越不可預測。
`max_tokens` 可限制回答長度上限，控制成本與延遲。

In [ ]:
strict_model = get_model(temperature=0)        # deterministic output
creative_model = get_model(temperature=1.0)    # more varied output

prompt = "幫一間賣咖啡的小店想一個店名"
print("temperature=0  :", strict_model.invoke(prompt).content)
print("temperature=1.0:", creative_model.invoke(prompt).content)
# Expected output: 兩種風格的店名；低溫較保守、高溫較跳

## 10. 查模型能力：model.profile

不確定這顆模型支不支援工具呼叫、看圖、結構化輸出？
查 `model.profile` 就好，不用翻文件、也不用試錯。

In [ ]:
# profile describes this model's capabilities as a dict-like object.
print(model.profile)
# Expected output: 一個描述模型能力的 dict（不同供應商欄位略有差異）

## 🧪 練習 1：加一則 AIMessage 延續對話

把第 5 格的 `conversation` 再延伸一輪：
1. 在最後補上模型對「第二定律」的回答（用 `AIMessage`，內容自己編一句）。
2. 再加一個 `HumanMessage` 問「那第三定律呢？」。
3. `invoke` 後印出 `.content`。

觀察：模型是否能根據前面的歷史，正確接著回答第三定律。

In [ ]:
# Your code here.
# Hint: conversation + [AIMessage("..."), HumanMessage("那第三定律呢？")]

## 🧪 練習 2：比較 temperature 對穩定性的影響

1. 用 `get_model(temperature=0)` 對同一個 prompt 連續 `invoke` 兩次。
2. 再用 `get_model(temperature=1.0)` 對同一個 prompt 連續 `invoke` 兩次。
3. 印出四個結果，觀察哪一組兩次比較接近（提示：低溫較可重現）。

In [ ]:
# Your code here.
# Hint: 重複呼叫並把 .content 印出來比較。

## 小結 & 下一步

你學會了：
- chat model 的統一介面：`invoke`（一次）/ `stream`（逐塊）/ `batch`（多筆）。
- 四種訊息物件與 `dict` 簡寫，用角色精準控制對話。
- content blocks 讓訊息能承載多模態（文字 + 圖片）。
- 用 `temperature` 調行為、用 `model.profile` 查模型能力。

下一個模組 **M02：PromptTemplate 與結構化輸出**：
每次手寫整串訊息太累又易錯，我們會用 `ChatPromptTemplate` 把訊息模板化，
並用 `with_structured_output` 強迫模型回傳乾淨的結構化資料，而不是一坨待解析的文字。